In [7]:
!pip install PyPDF2 ipywidgets

Defaulting to user installation because normal site-packages is not writeable


In [8]:
import PyPDF2
import re
import ipywidgets as widgets
from IPython.display import display, clear_output

In [9]:
def extract_text_from_pdf(uploaded_file):
    pdf_reader = PyPDF2.PdfReader(uploaded_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    return text.lower()

In [10]:
def calculate_score(resume_text, job_description):

    resume_words = re.findall(r'\b\w+\b', resume_text)
    job_words = re.findall(r'\b\w+\b', job_description.lower())

    matched_words = set(resume_words) & set(job_words)

    if len(set(job_words)) == 0:
        return 0, []

    score = (len(matched_words) / len(set(job_words))) * 100

    return round(score, 2), matched_words

In [11]:
upload = widgets.FileUpload(accept='.pdf', multiple=False)
jd_input = widgets.Textarea(
    placeholder='Paste Job Description Here...',
    description='Job Desc:',
    layout=widgets.Layout(width='100%', height='150px')
)

analyze_button = widgets.Button(description="Analyze Resume")
output = widgets.Output()

display(upload, jd_input, analyze_button, output)

FileUpload(value=(), accept='.pdf', description='Upload')

Textarea(value='', description='Job Desc:', layout=Layout(height='150px', width='100%'), placeholder='Paste Jo…

Button(description='Analyze Resume', style=ButtonStyle())

Output()

In [12]:
def on_button_click(b):
    with output:
        clear_output()

        if not upload.value:
            print("Please upload a resume PDF.")
            return

        if jd_input.value.strip() == "":
            print("Please enter Job Description.")
            return

        uploaded_file = upload.value[0]   # FIXED HERE
        content = uploaded_file['content']

        import io
        pdf_file = io.BytesIO(content)

        resume_text = extract_text_from_pdf(pdf_file)
        score, matched_words = calculate_score(resume_text, jd_input.value)

        print("🎯 Resume Score:", score, "%")
        print("\n✅ Matched Keywords:")
        print(matched_words)

        if score > 75:
            print("\nExcellent Resume! High chances 🎉")
        elif score > 50:
            print("\nGood Resume! Can improve more 👍")
        else:
            print("\nLow Matching Score ❌ Improve your resume.")

analyze_button.on_click(on_button_click)